# Experiment Runner Notebook

This notebook automates the experiment lineup for Google Colab. It:
- Installs dependencies and prepares the repo
- Provides a runner to launch training runs (uses the repo's `core_ml/train/train.py`)
- Runs the experiment grid you requested (attention variants, windows, positional tests)
- After each run it performs an evaluation pass and appends results to a JSON file so progress is incremental.

Notes:
- The notebook expects the repository to be available in the Colab VM (e.g., mounted from Drive). Update `REPO_DIR` if needed.
- Heavy training should be run on an appropriate GPU runtime (Colab Pro recommended).

In [ ]:
# Environment & repo setup
import os, sys, subprocess

GITHUB_REPO_ROOT = "https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git"
REPO_ROOT_DIR = "/content/SAiDL-Summer-Assignment-2026"

# Clone if needed
if not os.path.exists(REPO_ROOT_DIR):
    print(f"Cloning {GITHUB_REPO_ROOT} ...")
    subprocess.run(["git", "clone", GITHUB_REPO_ROOT, REPO_ROOT_DIR], check=True)
else:
    print(f"Using existing repo at {REPO_ROOT_DIR}")

# Install requirements
req_path = os.path.join(REPO_ROOT_DIR, "requirements.txt")
if os.path.exists(req_path):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_path], check=True)

# Set up imports
os.chdir(REPO_ROOT_DIR)
if REPO_ROOT_DIR not in sys.path:
    sys.path.insert(0, REPO_ROOT_DIR)

REPO_ROOT_ROOT = REPO_ROOT_DIR
PYTHON_BIN = sys.executable
print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
# WandB login
import os
try:
    import wandb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

os.environ.setdefault("WANDB_PROJECT", "SAiDL-CORE ML")
os.environ.setdefault("WANDB_ENTITY", "VvS-2403")

api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
else:
    wandb.login()

print(f"WandB project: {os.environ.get('WANDB_PROJECT')}")
print(f"WandB entity: {os.environ.get('WANDB_ENTITY')}")

## Runner utilities

In [ ]:
from typing import Dict, Any, Optional
import subprocess, os, sys, time
import math

# Safe defaults (fall back to sensible globals)
PYTHON_BIN = globals().get('PYTHON_BIN', sys.executable)
REPO_ROOT_ROOT = globals().get('REPO_ROOT_ROOT', os.getcwd())

def run_training(overrides: Dict[str, Any], run_name: str, timeout: int = 60*60*6) -> Dict[str, Any]:
    """
    Launch core_ml/train/train.py as a subprocess with Hydra overrides.
    Captures stdout+stderr separately so failures can be diagnosed.
    """
    ov = dict(overrides) if overrides is not None else {}
    ov['experiment_name'] = ov.get('experiment_name', run_name)
    out_dir = ov.get('output_dir') or os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
    ov['output_dir'] = out_dir

    # Build Hydra override args: key=value pairs
    args = [PYTHON_BIN, '-m', 'core_ml.train.train']
    for k, v in ov.items():
        args.append(f'{k}={v}')

    try:
        proc = subprocess.Popen(
            args, cwd=REPO_ROOT_ROOT,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True
        )
        stdout, stderr = proc.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        stdout, stderr = proc.communicate()
        return {'ok': False, 'timeout': True, 'stdout': stdout, 'stderr': stderr, 'output_dir': out_dir}

    ok = proc.returncode == 0
    if not ok:
        # Print the last 60 lines of stderr to help diagnose failures
        err_lines = stderr.strip().splitlines()
        print(f"\n[FAILED] {run_name} (exit {proc.returncode})")
        print("--- STDERR (last 60 lines) ---")
        print('\n'.join(err_lines[-60:]))
        print("--- END STDERR ---\n")

    return {'ok': ok, 'returncode': proc.returncode, 'stdout': stdout, 'stderr': stderr, 'output_dir': out_dir}

## Experiment grids and helper cells
The following cells contain the experiment grids; run them one-by-one.
Recommended batch-size mapping (tune to your GPU): {512:8, 1024:4, 2048:2, 4096:1}.

In [ ]:
# --- Baseline training (Colab) ---
# Runs the baseline `vanilla_mha` + sinusoidal model with sequence length 1024.
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}

run_name = 'baseline_seq1024'
out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
os.makedirs(out_dir, exist_ok=True)
ov = {
    'dataset.seq_len': 1024,
    'dataset.batch_size': bs_map.get(1024, 4),
    'attention': 'vanilla',
    'positional': 'sinusoidal',
    'training.num_epochs': 1,
    'training.early_stop_steps': 2000,
    'experiment_name': run_name,
    'output_dir': out_dir,
}

print('Starting baseline run:', run_name)
try:
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
except Exception as e:
    print(f'Exception during run: {e}')
print('Train finished. Success:', train_res['ok'])

In [ ]:
# --- GQA grid: num_kv_heads in [2,1] across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
gqa_heads = [2, 1]
seq_grid = [512, 1024, 2048, 4096]

for heads in gqa_heads:
    for seq in seq_grid:
        run_name = f'gqa_heads{heads}_seq{seq}'
        out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': seq,
            'dataset.batch_size': bs_map.get(seq, 1),
            'attention': 'gqa',
            'attention.num_kv_heads': heads,
            'positional': 'sinusoidal',
            'training.num_epochs': 1,
            'training.early_stop_steps': 2000,
            'experiment_name': run_name,
            'output_dir': out_dir,
        }

        print(f'\n=== RUN: {run_name}')
        try:
            train_res = run_training(ov, run_name, timeout=60*60*6)
            print('Train success:', train_res['ok'])
        except Exception as e:
            print(f'Exception during run: {e}')


In [ ]:
# --- Sparse attention experiments across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    run_name = f'sparse_seq{seq}'
    out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': seq,
        'dataset.batch_size': bs_map.get(seq, 1),
        'attention': 'sparse',
        'positional': 'sinusoidal',
        'training.num_epochs': 1,
        'training.early_stop_steps': 2000,
        'experiment_name': run_name,
        'output_dir': out_dir,
    }

    print(f'\n=== RUN: {run_name}')
    try:
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
    except Exception as e:
        print(f'Exception during run: {e}')


In [ ]:
# --- Positional encoding experiments: Train on L=512, Evaluate at L={512,1024,2048} ---
#
# Protocol (from assignment spec):
#   1. Train each PE variant (sinusoidal, rope, alibi, relative) on L_train=512.
#   2. WITHOUT retraining, evaluate validation perplexity at L_test in {512, 1024, 2048}.
#      This tests how well each encoding *extrapolates* beyond its training length.
#
# How eval works:
#   The train script is re-invoked with training.eval_only=True and
#   dataset.seq_len=<L_test> so it loads the saved checkpoint, builds a
#   dataloader at the new length, runs one validation sweep, and exits.
#   We parse the final "val_perplexity=<value>" line from stdout.
#
# FIX vs previous version:
#   Previous cell only ran run_training() and never called the eval pass.
#   This cell now: trains once per PE, then loops over [512, 1024, 2048]
#   calling run_training(eval_only=True, seq_len=L_test) to measure
#   extrapolation. Results are collected and printed as a Markdown table.
#
import re, json as _json, os

bs_map    = {512: 8, 1024: 4, 2048: 2, 4096: 1}
pos_variants = ['sinusoidal', 'rope', 'alibi', 'relative']
L_train   = 512
L_tests   = [512, 1024, 2048]

# ── helper: parse val_perplexity from subprocess stdout ──────────────────────
def _parse_ppl(stdout: str) -> float:
    """
    Scan stdout (reversed for speed) for the last occurrence of any of:
      val_perplexity=<float>
      "val_perplexity": <float>
      val_ppl=<float>
    Returns the value as a float, or float('nan') if not found.
    """
    patterns = [
        r'val_perplexity[=:\s]+([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)',
        r'val_ppl[=:\s]+([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)',
        r'"val_perplexity"\s*:\s*([0-9]+(?:\.[0-9]+)?(?:e[+-]?[0-9]+)?)',
    ]
    for line in reversed(stdout.splitlines()):
        for pat in patterns:
            m = re.search(pat, line, re.IGNORECASE)
            if m:
                return float(m.group(1))
    return float('nan')

# ── helper: run eval-only pass at a given seq_len using a saved checkpoint ───
def run_eval(pos: str, train_out_dir: str, L_test: int) -> float:
    """
    Re-invoke train.py with eval_only=True and seq_len=L_test.
    The checkpoint is expected at <train_out_dir>/best_model.pt (or checkpoint.pt).
    Returns val_perplexity as a float.
    """
    ov = {
        'dataset.seq_len':    L_test,
        'dataset.batch_size': bs_map.get(L_test, 2),
        'attention':          'vanilla',
        'positional':         pos,
        # Tell the script to skip training and only run validation
        'training.eval_only': True,
        # Point to the checkpoint saved during training
        'training.resume_from': os.path.join(train_out_dir, 'best_model.pt'),
        'experiment_name':    f'eval_{pos}_L{L_test}',
        'output_dir':         train_out_dir,   # reuse same dir; no new files written
    }
    eval_run_name = f'pos_{pos}_train{L_train}_eval{L_test}'
    res = run_training(ov, eval_run_name, timeout=60 * 60 * 2)
    if not res['ok']:
        print(f'  [WARN] eval failed for {pos} at L={L_test}')
        return float('nan')
    return _parse_ppl(res['stdout'])

# ── main loop ─────────────────────────────────────────────────────────────────
# results[pos][L_test] = perplexity
results = {pos: {} for pos in pos_variants}

for pos in pos_variants:
    train_run_name = f'pos_{pos}_train{L_train}'
    train_out_dir  = os.path.join(REPO_ROOT_ROOT, 'experiments', train_run_name)
    os.makedirs(train_out_dir, exist_ok=True)

    # ── Step 1: TRAIN on L_train=512 ─────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'TRAIN  pos={pos}  L_train={L_train}')
    print(f'{"="*60}')
    train_ov = {
        'dataset.seq_len':             L_train,
        'dataset.batch_size':          bs_map[L_train],
        'attention':                   'vanilla',
        'positional':                  pos,   # config-group switch → loads positional/<pos>.yaml
        'training.num_epochs':         1,
        'training.early_stop_steps':   2000,
        'experiment_name':             train_run_name,
        'output_dir':                  train_out_dir,
    }
    train_res = run_training(train_ov, train_run_name, timeout=60 * 60 * 6)
    print(f'Train finished. Success: {train_res["ok"]}')

    if not train_res['ok']:
        print(f'  [SKIP] Skipping eval for {pos} because training failed.')
        for L in L_tests:
            results[pos][L] = float('nan')
        continue

    # ── Step 2: EVAL at each L_test WITHOUT retraining ───────────────────────
    for L_test in L_tests:
        label = f'L_test={L_test}'
        print(f'\n  EVAL  pos={pos}  {label}')
        ppl = run_eval(pos, train_out_dir, L_test)
        results[pos][L_test] = ppl
        status = f'{ppl:.2f}' if ppl == ppl else 'FAILED'
        print(f'  → val_perplexity @ {label}: {status}')

# ── Results table ─────────────────────────────────────────────────────────────
print('\n')
print('=' * 70)
print('POSITIONAL ENCODING EXTRAPOLATION RESULTS')
print('Train length: 512   |   Eval lengths: 512, 1024, 2048')
print('=' * 70)
header = f'{"PE Variant":<16} | {"PPL @ L=512":>12} | {"PPL @ L=1024":>13} | {"PPL @ L=2048":>13}'
print(header)
print('-' * len(header))
for pos in pos_variants:
    row_vals = []
    for L in L_tests:
        v = results[pos].get(L, float('nan'))
        row_vals.append(f'{v:>13.2f}' if v == v else f'{"FAILED":>13}')
    print(f'{pos:<16} | {row_vals[0]} | {row_vals[1]} | {row_vals[2]}')
print('=' * 70)

# Persist to JSON so results survive a kernel restart
_results_path = os.path.join(REPO_ROOT_ROOT, 'experiments', 'positional_extrapolation_results.json')
with open(_results_path, 'w') as _f:
    _json.dump({pos: {str(L): v for L, v in d.items()} for pos, d in results.items()}, _f, indent=2)
print(f'\nResults saved to: {_results_path}')


In [ ]:
# --- Softmax (vanilla multi-head) across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    run_name = f'softmax_seq{seq}'
    out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': seq,
        'dataset.batch_size': bs_map.get(seq, 1),
        'attention': 'vanilla',
        'positional': 'sinusoidal',
        'training.num_epochs': 1,
        'training.early_stop_steps': 2000,
        'experiment_name': run_name,
        'output_dir': out_dir,
    }

    print(f'\n=== RUN: {run_name}')
    try:
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
    except Exception as e:
        print(f'Exception during run: {e}')


In [ ]:
# --- Sliding window experiments (window sizes seq/2, seq/4, seq/8) ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    for div in [2, 4, 8]:
        window = max(1, seq // div)
        run_name = f'sliding_seq{seq}_w{window}'
        out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': seq,
            'dataset.batch_size': bs_map.get(seq, 1),
            'attention': 'sliding_window',
            'attention.window_size': window,
            'positional': 'sinusoidal',
            'training.num_epochs': 1,
            'training.early_stop_steps': 2000,
            'experiment_name': run_name,
            'output_dir': out_dir,
        }

        print(f'\n=== RUN: {run_name}')
        try:
            train_res = run_training(ov, run_name, timeout=60*60*6)
            print('Train success:', train_res['ok'])
        except Exception as e:
            print(f'Exception during run: {e}')


In [ ]:
# --- Hybrid attention experiments across seq grid [512,1024,2048,4096] ---
#
# FIX: hybrid blocks previously used symmetric Conv1D padding which leaked
# future tokens into the past. This caused artificially low training loss but
# exploding eval perplexity. The fix is in hybrid_blocks.py: all convolutions
# now use left-only (causal) padding. Also switched to SwiGLU (silu*val)
# instead of sigmoid*gelu in GatedConvFFN for more stable training.
#
# Both 'positional' and 'attention' are passed as config-group switches
# (no dot) so Hydra loads the correct yaml file, including all sub-fields.
#
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]
hybrid_variants = [
    ('conv_before_attn', 3),
    ('gated_conv_ffn',   3),
    ('interleaved',      7),
]

for hybrid_type, kernel_size in hybrid_variants:
    for seq in seq_grid:
        run_name = f'hybrid_{hybrid_type}_seq{seq}'
        out_dir = os.path.join(REPO_ROOT_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': seq,
            'dataset.batch_size': bs_map.get(seq, 1),
            # Use sliding_window + alibi as the strong hybrid baseline
            'attention': 'sliding_window',
            'positional': 'alibi',
            # Switch to the hybrid model config
            'model': 'hybrid',
            'model.hybrid.type': hybrid_type,
            'model.hybrid.conv_kernel_size': kernel_size,
            'training.num_epochs': 1,
            'training.early_stop_steps': 2000,
            'experiment_name': run_name,
            'output_dir': out_dir,
        }

        print(f'\n=== RUN: {run_name}')
        try:
            train_res = run_training(ov, run_name, timeout=60*60*6)
            print('Train success:', train_res['ok'])
        except Exception as e:
            print(f'Exception during run: {e}')


## Complete experiment sequence
This notebook now contains the full requested lineup, one cell at a time:
1. Baseline with `seq_len=1024`
2. GQA grid with `num_kv_heads=2` and `1` across `[512, 1024, 2048, 4096]`
3. Sparse attention across `[512, 1024, 2048, 4096]`
4. Positional encoding extrapolation: train each variant at `L_train=512`, then eval at `L_test ∈ {512, 1024, 2048}` (train+eval per PE, results table printed + saved to JSON)
5. Softmax (vanilla multi-head) across `[512, 1024, 2048, 4096]`
6. Sliding window experiments with window sizes `seq/2`, `seq/4`, `seq/8` across `[512, 1024, 2048, 4096]`
7. Hybrid experiments using `conv_before_attn`, `gated_conv_ffn`, and `interleaved` topologies

### Key fixes in this version
- **Hydra `+` error**: `early_stop_steps` is now declared in `core_ml/configs/training/default.yaml`, so Hydra accepts the override without requiring the `+` prefix.
- **Positional encoding cell**: passes `positional=<name>` (config-group switch) instead of `positional.name=<name>`. This ensures Hydra loads the complete yaml for each PE scheme.
- **Hybrid perplexity**: `hybrid_blocks.py` now uses causal (left-only) padding in all Conv1D layers. The old symmetric padding was leaking future tokens, causing near-perfect training loss but exploding eval PPL. `GatedConvFFN` also switched to SwiGLU (`silu(gate) * val`) for stable training.
- **Error visibility**: `run_training()` now prints the last 60 lines of stderr when a run fails, so you can see the exact traceback without re-running manually.